# Activité 5 — Agents et assistance SAV

Le projet prend les notebooks du cours et la [démo MCP du professeur](https://github.com/mohamedYoussfi/agent-rag-as-tool-mcp-v2) comme références. Le manuel PX-200, ses figures et le registre de garantie ont été créés pour la démonstration.

### Préparer VS Code

1. Ouvrir le dossier `activite-5` avec **Fichier → Ouvrir un dossier**.
2. Choisir **Terminal → Nouveau terminal** et lancer la commande ci-dessous. Elle installe les dépendances dans le dossier du projet.

```bash
"$HOME/.local/bin/uv" sync --locked
```

3. Ouvrir ce notebook. Cliquer sur **Sélectionner le noyau → Environnements Python** et choisir le Python 3.11 du projet, associé à `.venv/bin/python`.
4. Dans le terminal de VS Code, démarrer le serveur MCP. Garder ce terminal ouvert pour les sections C et D.

```bash
"$HOME/.local/bin/uv" run python -m sav.serveur_mcp
```

Les sorties enregistrées couvrent les démonstrations reproductibles. Les cellules facultatives se sélectionnent individuellement pour utiliser Ollama ou le Web.

## A. LangChain

Le premier agent reçoit une question. Le second conserve les messages dans une mémoire associée à un `thread_id`. Les réponses fixées ci-dessous permettent d’observer l’appel d’outil, le changement de profil et la demande d’approbation à chaque exécution.

In [1]:
from langgraph.types import Command
from sav.langchain_demo import Contexte, agent_simple, creer_agent, python_repl_calcul
from test_activite import ModeleDeTest, action, message

simple = agent_simple(ModeleDeTest(messages=iter(["Bonjour, comment puis-je vous aider ?"])))
print(simple.invoke(message("Bonjour."))["messages"][-1].content)

base = ModeleDeTest(messages=iter([
    action("calculer_ttc", {"prix_ht": 100}), "Le prix TTC est 120 €.",
    action("calculer_ttc", {"prix_ht": -1}, "prix-invalide"), "Le prix est invalide.",
    action("ouvrir_ticket_demo", {"sujet": "Voyant orange"}, "ticket"), "Ticket traité.",
]))
expert = ModeleDeTest(messages=iter(["Régime d’exemple : TVA de 20 %."]))
agent = creer_agent(base, expert)
fil = {"configurable": {"thread_id": "atelier-a"}}

def poser(question, niveau="debutant"):
    return agent.invoke(message(question), fil, context=Contexte(niveau))

prix = poser("Quel est le prix TTC pour 100 € ?")
print("Observation TTC =", next(m.content for m in prix["messages"] if m.type == "tool"))
print("Profil expert =", poser("Détaille le calcul.", "expert")["messages"][-1].content)
print("Tours en mémoire =", sum(m.type == "human" for m in agent.get_state(fil).values["messages"]))
erreur = poser("Quel est le prix TTC pour −1 € ?")
print("Erreur traitée =", next(m.content for m in erreur["messages"] if getattr(m, "status", None) == "error"))
print("Garde-fou =", poser("x" * 2001)["messages"][-1].content)
pause = poser("Ouvre un ticket pour le voyant orange.")
print("Approbation demandée =", bool(pause["__interrupt__"]))
resume = Command(resume={"decisions": [{"type": "approve"}]})
fin = agent.invoke(resume, fil, context=Contexte())
print("Après approbation =", next(m.content for m in fin["messages"] if getattr(m, "name", None) == "ouvrir_ticket_demo"))
print("PythonREPLTool =", python_repl_calcul.invoke({"expression": "(7+2)*3"}))

Bonjour, comment puis-je vous aider ?
Observation TTC = 120.0
Profil expert = Régime d’exemple : TVA de 20 %.
Tours en mémoire = 2
Erreur traitée = L’outil a échoué. Vérifie les paramètres et la configuration.
Garde-fou = Retire les données sensibles ou raccourcis la demande.
Approbation demandée = True
Après approbation = Ticket simulé approuvé pour « Voyant orange ». Aucun ticket externe créé.
PythonREPLTool = 27


Pour essayer les agents avec Ollama, utiliser les modèles locaux `qwen2.5:1.5b` et `llama3.2:3b`. Dans un autre terminal, `ollama list` montre les modèles disponibles. Utiliser `ollama pull NOM_DU_MODELE` si l’un d’eux est absent.

DuckDuckGo et Tavily sont des outils facultatifs. La recherche publique peut être activée avec `creer_agent(web=True)`. Tavily lit la variable d’environnement `TAVILY_API_KEY`.

In [ ]:
from sav.langchain_demo import chercher_duckduckgo, chercher_tavily, chatbot_rag_langchain

ESSAYER_OLLAMA = False
ESSAYER_WEB = False
if ESSAYER_OLLAMA:
    simple = agent_simple()
    print(simple.invoke(message("Présente ton rôle en une phrase."))["messages"][-1].content)
    discussion = chatbot_rag_langchain()
    config = {"configurable": {"thread_id": "rag-a"}, "recursion_limit": 12}
    sortie = discussion.invoke(message("Cherche la figure img_px200_01 du PX-200 et cite sa page."), config)
    print(sortie["messages"][-1].content)
if ESSAYER_WEB:
    import os
    print(chercher_duckduckgo.invoke({"requete": "documentation officielle LangGraph"}))
    if os.getenv("TAVILY_API_KEY"):
        print(chercher_tavily.invoke({"requete": "documentation officielle LangChain"}))

## B. LangGraph, Studio et Streamlit

Le graphe de base montre un état, un nœud de normalisation et une branche conditionnelle. Le chatbot laisse le modèle choisir l’outil et repasse par le modèle après chaque observation. Si le modèle répond à une question documentaire explicite sans lancer de recherche, le graphe consulte le manuel. Si la réponse manque de citations, il affiche les passages retrouvés avec leurs références.

In [3]:
import json
from sav.graphes import graphe_agentique, graphe_bases

bases = graphe_bases()
for entree in (" Bonjour ", "Un voyant clignote."):
    print([list(etape) for etape in bases.stream({"entree": entree}, stream_mode="updates")])

demo = ModeleDeTest(messages=iter([
    action("retriever_tool", {"question": "voyant orange papier", "image_ref": "img_px200_01"}),
    "Vérifiez le bac papier [PX200-T01, p. 4].",
]))
chatbot = graphe_agentique(demo, memoire=False)
sortie = chatbot.invoke(message("Que signifie ce voyant ?"))
for m in sortie["messages"][1:]:
    if getattr(m, "tool_calls", None):
        print("Action =", m.tool_calls[0]["name"])
    elif m.type == "tool":
        print("Observation =", [(s["id"], s["page"], s["type"]) for s in json.loads(m.content)])
    else:
        print("Réponse de test =", m.content)

[['normaliser'], ['saluer']]
[['normaliser'], ['orienter']]
Action = retriever_tool
Observation = [('PX200-F01', 4, 'figure'), ('PX200-T01', 4, 'texte')]
Réponse de test = Vérifiez le bac papier [PX200-T01, p. 4].


### Tester dans Studio

Ouvrir un nouveau terminal dans VS Code. Installer le groupe Studio, puis lancer le serveur de développement.

```bash
"$HOME/.local/bin/uv" sync --locked --group studio
"$HOME/.local/bin/uv" run langgraph dev --port 2025
```

Ouvrir le lien **Studio UI** affiché dans le terminal. Choisir `bases` avec `{"entree":" Bonjour "}` ou `rag` avec `{"question":"Quel est le voyant orange sur mon PX-200 ?","image_ref":"img_px200_01"}`. Inspecter les nœuds parcourus. Le graphe `agentique` accepte un état `messages` et utilise Ollama. Les graphes `bases` et `rag` ont également été invoqués avec succès par l’API locale de LangGraph.

### Ouvrir le chatbot web

Dans un autre terminal, lancer la commande ci-dessous et ouvrir l’adresse locale affichée.

```bash
"$HOME/.local/bin/uv" run streamlit run app.py
```

Le menu de gauche propose **Chatbot agentique LangGraph** avec Ollama et **SAV multimodal et garantie** avec le serveur MCP.

## C. MCP via Streamable HTTP

Le serveur expose `rechercher_manuel` pour récupérer des sources et `rag_multimodal` pour obtenir une réponse accompagnée de ses sources. L’adresse locale est `http://127.0.0.1:8005/mcp`.

Avec Node.js installé, lancer l’Inspector depuis un autre terminal.

```bash
npx -y @modelcontextprotocol/inspector
```

Dans l’Inspector, choisir **Streamable HTTP** et coller l’adresse MCP. Cliquer sur **Connect**, puis **Tools → List Tools**. Choisir `rag_multimodal`, entrer `Quel voyant sur mon PX-200 ?` dans **question** et `img_px200_01` dans **image_ref**. Le test CLI de l’Inspector a listé les deux outils et l’appel de `rag_multimodal` a renvoyé les sources PX200-F01 et PX200-T01.

In [4]:
from sav.client_mcp import agent_mcp, appeler_mcp

documents = await appeler_mcp("rechercher_manuel", {
    "question": "Quel voyant sur mon PX-200 ?", "image_ref": "img_px200_01",
})
print("Client MCP =", [(s["id"], s["page"]) for s in documents])

demo_mcp = ModeleDeTest(messages=iter([
    action("rag_multimodal", {"question": "voyant orange", "image_ref": "img_px200_01"}),
    "Figure du bac papier [PX200-F01, p. 4].",
]))
client_agent = await agent_mcp(demo_mcp)
sortie = await client_agent.ainvoke(message("Que signifie img_px200_01 ?"))
print("Outil distant appelé =", next(m.name for m in sortie["messages"] if m.type == "tool"))
print("Réponse de test =", sortie["messages"][-1].content)

Client MCP = [('PX200-F01', 4), ('PX200-T01', 4)]
Outil distant appelé = rag_multimodal
Réponse de test = Figure du bac papier [PX200-F01, p. 4].


Pour lancer le client LangChain avec Ollama, utiliser `"$HOME/.local/bin/uv" run python -m sav.client_mcp`.

### Importer le workflow n8n

1. Dans n8n sur le même Mac, créer un workflow et choisir **Import from File**. Sélectionner `n8n_sav_mcp.json`.
2. Ouvrir **Modèle Ollama** et sélectionner ou créer une connexion vers `http://127.0.0.1:11434`. Le modèle préconfiguré est `llama3.2:3b`.
3. Ouvrir **MCP Client** et vérifier le transport **HTTP Streamable** ainsi que l’adresse `http://127.0.0.1:8005/mcp`. Ces adresses supposent que n8n tourne directement sur le Mac.
4. Avec MCP et Ollama actifs, ouvrir **Chat** et demander « Que signifie le voyant image img_px200_01 du PX-200 ? ». Dans l’exécution, vérifier l’appel de `rag_multimodal` et les sources.

## D. Synthèse SAV

L’orchestrateur choisit une seule catégorie. Le spécialiste documentaire interroge le serveur MCP. Le spécialiste garantie lit le registre local. Le graphe repasse ensuite par l’orchestrateur, qui termine dès qu’une réponse est disponible.

La recherche documentaire compare les mots de la question et, le cas échéant, les caractéristiques visuelles de la photo avec les figures de démonstration. La réponse assemble les passages récupérés et indique leur page.

In [5]:
from sav.orchestrateur import sav_graph

exemples = [
    {"question": "Mon imprimante PX-200 affiche un voyant clignotant. Que signifie le pictogramme sur la photo jointe ?",
     "image_ref": "img_px200_01"},
    {"question": "Est-ce que mon imprimante est encore sous garantie ? Son numéro de série est PX200-0042."},
]
for entree in exemples:
    sortie = {}
    trajet = []
    async for etape in sav_graph.astream(entree, stream_mode="updates"):
        noeud, changements = next(iter(etape.items()))
        trajet.append(noeud)
        sortie.update(changements or {})
    print(" → ".join(trajet))
    print(sortie["categorie"], "·", sortie["agent"])
    print(sortie["reponse"])
    print()

orchestrateur → agent_documentaire → orchestrateur
documentation · agent_documentaire
Figure 1. Voyant orange clignotant avec une feuille : incident d’alimentation papier. [PX200-F01, p. 4]

Voyant orange clignotant et pictogramme de feuille : vérifier le bac papier. Retirer le papier coincé sans forcer, remettre le bac et appuyer une fois sur Reprendre. [PX200-T01, p. 4]

orchestrateur → agent_garantie → orchestrateur
garantie · agent_garantie
La garantie de PX200-0042 est active jusqu’au 2027-08-14 inclus.



### Relancer les vérifications

Avec le serveur MCP actif, ouvrir un nouveau terminal depuis le projet.

```bash
TEST_MCP=1 "$HOME/.local/bin/uv" run python -m unittest -v test_activite
```

Les tests contrôlent notamment la mémoire, les middlewares, le routage des deux demandes, la communication HTTP avec le serveur et les deux scénarios dans Streamlit.